# Building a Simple AI Fashion Assistant

In [1]:
import os
import json
from dotenv import load_dotenv
from openai import OpenAI
import gradio as gr

## 2. Environment Setup — API Keys & LLM Clients

In [2]:
load_dotenv(override=True)

openrouter_api_key= os.getenv("OPENROUTER_API_KEY")
gemini_api_key = os.getenv("GEMINI_API_KEY")
ollama_api_key = "ollama"


openrouter_url = "https://openrouter.ai/api/v1"
gemini_url= "https://generativelanguage.googleapis.com/v1beta/openai/"
ollama_url= "http://localhost:11434/v1"

gemini = OpenAI(api_key=gemini_api_key, base_url=gemini_url)
openrouter = OpenAI(base_url=openrouter_url, api_key=openrouter_api_key)
ollama = OpenAI(api_key=ollama_api_key, base_url=ollama_url)


gemini_model = "gemini-3.6-flash"
gpt_model = "openai/gpt-4o-mini"
openrouter_model="openrouter/free"
ollama_model = "llama3.2"

## 3. Brand Data — Website Pages & URLs

In [3]:
lumaa_pages = {

    "home": "https://lumaatunnoor.com/",
    "sale": "https://lumaatunnoor.com/sale/",
    "lumaa brides": "https://lumaatunnoor.com/lumaa-brides/",
    "bridals": "https://lumaatunnoor.com/bridals/",
    "luxury formals": "https://lumaatunnoor.com/luxury-formals/",
    "luxury lawn": "https://lumaatunnoor.com/luxury-lawn/",
    "menswear": "https://lumaatunnoor.com/menswear/",

    #Shop by occasion
 
    "mehndi":
        "https://lumaatunnoor.com/shop-by-occasions/mehndi/",
    "baraat":
        "https://lumaatunnoor.com/shop-by-occasions/baraat/",
    "walima":
        "https://lumaatunnoor.com/shop-by-occasions/walima/",
    "nikkah":
        "https://lumaatunnoor.com/shop-by-occasions/nikkah/",
    "eid":
        "https://lumaatunnoor.com/shop-by-occasions/eid/",
    "wedding":
        "https://lumaatunnoor.com/shop-by-occasions/wedding/",
    "festive":
        "https://lumaatunnoor.com/shop-by-occasions/festive/",

    # Shop by style
 
    "gharara":
        "https://lumaatunnoor.com/shop-by-style/gharara/",
    "lehenga":
        "https://lumaatunnoor.com/shop-by-style/lehenga/",
    "pishwas":
        "https://lumaatunnoor.com/shop-by-style/pishwas/",
    "saree":
        "https://lumaatunnoor.com/shop-by-style/saree/",
    "sharara":
        "https://lumaatunnoor.com/shop-by-style/sharara/",
    "shirt trousers":
        "https://lumaatunnoor.com/shop-by-style/shirt-trousers/",
    "lehenga choli":
        "https://lumaatunnoor.com/shop-by-style/lehenga-choli/",

    # Policies / information
    "refunds":
        "https://lumaatunnoor.com/refund_returns/",
    "payment methods":
        "https://lumaatunnoor.com/payment-methods/",
    "return exchange":
        "https://lumaatunnoor.com/return-exchange/",
    "terms conditions":
        "https://lumaatunnoor.com/terms-condition/"
}

## 4. Brand Data — Contact Information

In [4]:
lumaa_contact = {

    "email": "support@lumaatunnoor.com",

    "phone": "+923104992111"
}


## 5. Intent Detection — Style Keywords

In [5]:
def detect_style(message):
        message = message.lower()
        styles = ["lehenga choli","shirt trousers","pishwas","lehenga","gharara","saree","sharara"]
        for style in styles:
         if style in message:
            return style
        return None


## 6. Intent Detection — Occasion Keywords

In [6]:
def detect_occasion(message):
    message = message.lower()
    occasions = ["mehndi","baraat","walima","nikkah","eid","wedding","festive"
    ]
    for occasion in occasions:
        if occasion in message:
            return occasion
    return None

## 7. Intent Detection — Collection/Category Keywords

In [7]:
def detect_category(message):
    message = message.lower()
    categories = ["luxury formals","luxury lawn","lumaa brides","menswear","bridals","sale"]
    for category in categories:
        if category in message:
            return category
    return None

## 8. Intent Detection — Policy Keywords (Refunds, Returns, Payment)

In [8]:
def detect_policy(message):
    message = message.lower()
    # Refund related words
    if "refund" in message:
        return "refunds"

    # Return / exchange related words
    if (
        "return" in message
        or "exchange" in message
        or "size exchange" in message
    ):
        return "return exchange"

    # Payment related words
    if (
        "payment" in message
        or "pay" in message
        or "payment method" in message
    ):
        return "payment methods"

    # Terms and conditions
    if (
        "terms" in message
        or "conditions" in message
    ):
        return "terms conditions"

        return None

## 9. Intent Detection — Contact Request

In [9]:
def detect_contact_request(message):
    message = message.lower()
    contact_words = ["contact","phone","number","email","support","customer service"]

    for word in contact_words:
        if word in message:
            return True
    return False


## 10. URL Resolver — Map Keyword to Page URL

In [10]:
def find_lumaa_url(keyword):
    # Make sure the keyword is lowercase.
    keyword = keyword.lower().strip()
    # Look for the keyword in our Lumaa dictionary.
    if keyword in lumaa_pages:
        return lumaa_pages[keyword]
    # If there is no matching page:
    return None

## 11. Context Builder — Orchestrating All Detectors

In [11]:
def build_lumaa_context(message):
    # Start with an empty context.
    context = ""
   
    # Check for a clothing style
    style = detect_style(message)
    if style:
        url = find_lumaa_url(style)
        if url:
            context += f"""The customer mentioned the Lumaa style:{style}
            Official Lumaa URL:{url}"""
  
    # Check for an occasion
    occasion = detect_occasion(message)
    if occasion:
        url = find_lumaa_url(occasion)
        if url:

            context += f"""The customer mentioned the Lumaa occasion:{occasion}
            Official Lumaa URL:{url}"""

    # Check for a main category
    category = detect_category(message)
    if category:
        url = find_lumaa_url(category)
        if url:
            context += f"""The customer mentioned the Lumaa collection/category:{category}
            Official Lumaa URL:{url}"""

    # Check for a policy request
    policy = detect_policy(message)
    if policy:
        url = find_lumaa_url(policy)
        if url:
            context += f"""The customer is asking about:{policy}
            Official Lumaa URL: {url}"""


    # Check for contact information

    contact_requested = detect_contact_request(message)
    if contact_requested:
        context += f"""Official Lumaa contact information:
        Email:{lumaa_contact["email"]}
        Phone:{lumaa_contact["phone"]}
        """
        
    # Return everything we discovered

    if not context.strip():
       context = (
        "Remind the customer you are Lumaa's AI assistant and can help them "
        "explore collections like bridals, luxury formals, luxury lawn, "
        "festive wear, menswear, and shop-by-occasion categories such as "
        "mehndi, baraat, walima, nikkah, eid, and wedding. " 
        "or shop-by-style categories such as lehenga, pishwas, gharara, sharara, saree, shirt & Trousers, Maxi etc."
        "Direct them to the Lumaa homepage: https://lumaatunnoor.com/"
    )
    return context

## 12. System Prompt — LLM Persona & Guardrails

In [12]:
system_message = """
You are Lumaa's AI fashion shopping assistant.
Lumaa is a Pakistani fashion brand specializing in women's
bridal, formal, luxury lawn, festive collections and menswear.

============================================================
YOUR ONLY JOB
============================================================
Answer ONLY using information provided to you in the context
by the Python system.
If the context is empty or does not contain the answer,
say you don't have that information and suggest the customer
visit https://lumaatunnoor.com/ or contact Lumaa directly.

============================================================
STRICT RULES — NEVER BREAK THESE
============================================================
- NEVER ask about budget, price range, or affordability.
  You have no pricing data. Do not pretend you do.
- NEVER suggest a collection, style, or occasion unless its
  official URL was explicitly provided in the context.
- NEVER invent or guess prices, stock levels, size
  availability, or product details.
- NEVER modify, shorten, or guess a Lumaa URL.
  Only use the exact URLs provided in the context.
- NEVER recommend a page using logic like "this might fit
  your budget" or "this is probably available".
- Do NOT ask clarifying questions about budget or price.
- NEVER recommend size and color.
- NEVER discuss style other than what is described in shop-by-style category.
- DO NOT ask questions about fabric or fabric-related details.

============================================================
CLARIFYING QUESTIONS — STRICT LIMITS
============================================================
You may ONLY ask the customer to clarify if their answer
would be one of these exact detectable keywords:

OCCASIONS: mehndi, baraat, walima, nikkah, eid, wedding, festive
STYLES:    lehenga, pishwas, gharara, sharara, saree,
           shirt trousers, lehenga choli
CATEGORIES: bridals, luxury formals, luxury lawn,
            lumaa brides, menswear, sale

Example of a GOOD clarifying question:
"Is this for a mehndi, baraat, walima, or nikkah?"

NEVER ask about:
- budget or price
- "traditional vs modern"
- colors or embellishment style
- size or measurements
- anything not in the list above

If you already have enough to share a URL from the context,
just share it. Do not ask unnecessary follow-up questions.

============================================================
WHAT YOU CAN DO
============================================================
- Share the official Lumaa URL(s) provided in the context.
- Describe the type of collection the URL leads to (e.g.,
  wedding occasion wear, lehenga styles).
- Suggest the customer contact Lumaa for anything beyond
  your available information:
  Email: support@lumaatunnoor.com | Phone: +923104992111
- Keep responses short, warm, and on-brand.


============================================================
PERSONALITY
============================================================
- Helpful, concise, polite, fashion-aware
- Do NOT sound like a generic AI chatbot
- Do NOT ramble or pad responses with guesses
"""


## 13. Chat Function — Context Injection + Streaming

In [13]:
def chat(message, history):
    history = [{"role": h["role"], "content": h["content"]} for h in history]

    lumaa_context = build_lumaa_context(message)

    messages = ([{"role": "system","content": system_message},
        {"role": "system","content": f"""Here is additional Lumaa information relevant to the 
        customer's current message:{lumaa_context}"""}]
    + history+ [{"role": "user", "content": message}])

    stream = openrouter.chat.completions.create(
        model=openrouter_model,
        messages=messages,
        stream=True
    )

    response = ""

    for chunk in stream:
        response += chunk.choices[0].delta.content or ""
        yield response

## 14. Launch — Gradio UI

In [14]:
demo = gr.ChatInterface(
    fn=chat,
    title="Lumaa AI Fashion Assistant",
    description=(
        "Ask me about Lumaa collections, "
        "styles, occasions and website information."
    )
)
demo.launch()

* Running on local URL:  http://127.0.0.1:7860
* To create a public link, set `share=True` in `launch()`.
